# Piper Castor — treinamento, continuação e exportação

Notebook organizado para treinar a voz **Castor** no Google Colab usando GPU NVIDIA e um checkpoint brasileiro do Piper como ponto inicial.

## Antes de executar

1. Em **Ambiente de execução → Alterar o tipo de ambiente de execução**, selecione uma GPU (T4 ou melhor).
2. No Google Drive, crie `Meu Drive/piper-castor/`.
3. Coloque nessa pasta o arquivo `castor_dataset.zip`.
4. O ZIP deve conter um `metadata.csv` e 1100 WAVs, mesmo que estejam em uma subpasta.
5. Execute as células de cima para baixo.

> Tudo que está em `/content` é temporário. Checkpoints, configuração e modelo final são gravados no Google Drive.


## 1. Configuração central

Este é o principal bloco que você poderá alterar. As opções mais comuns estão marcadas com `ALTERE AQUI`.


In [ ]:
from pathlib import Path
import os

# ========================= ALTERE AQUI =========================
DRIVE_FOLDER = "piper-castor"
DATASET_ZIP_NAME = "castor_dataset.zip"
VOICE_NAME = "pt_BR-castor-medium"

# Quantas épocas NOVAS serão executadas nesta rodada.
ADDITIONAL_EPOCHS = 30

# T4: comece com 8. Se ocorrer "CUDA out of memory", use 4.
BATCH_SIZE = 8

# Salva no Drive a cada N épocas. Valores menores protegem mais contra
# desconexões, mas escrevem arquivos grandes com mais frequência.
CHECKPOINT_EVERY_N_EPOCHS = 5

# True: continua do last.ckpt mais recente, se existir.
# False: começa novamente a partir do checkpoint Faber preparado.
RESUME_TRAINING = True

TEST_TEXT = "Olá, meu nome é Castor. Este é o meu novo modelo de voz."

# Parâmetros de síntese. Reduza noise_scale/noise_w_scale se a voz ficar
# instável, metálica ou com ritmo muito irregular.
TEST_NOISE_SCALE = 0.50
TEST_LENGTH_SCALE = 1.00
TEST_NOISE_W_SCALE = 0.60
# ===============================================================

BASE_CHECKPOINT_URL = (
    "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/"
    "pt/pt_BR/faber/medium/epoch%3D6159-step%3D1230728.ckpt?download=true"
)
BASE_CHECKPOINT_SHA256 = (
    "9a7a3817b079ac7786782cee14a3deb739b8cc5bfd3c08ef3abb62b5cffe712a"
)

CONTENT_DIR = Path("/content")
DRIVE_MOUNT = CONTENT_DIR / "drive"
DRIVE_DIR = DRIVE_MOUNT / "MyDrive" / DRIVE_FOLDER
DATASET_ZIP = DRIVE_DIR / DATASET_ZIP_NAME
DATASET_DIR = CONTENT_DIR / "castor_dataset"
AUDIO_DIR = DATASET_DIR / "audio"
METADATA_PATH = DATASET_DIR / "metadata.csv"
CACHE_DIR = CONTENT_DIR / "castor_cache"
PIPER_DIR = CONTENT_DIR / "piper1-gpl"
CHECKPOINT_DIR = CONTENT_DIR / "checkpoints"
TRAINING_DIR = DRIVE_DIR / "training"
MODEL_PATH = DRIVE_DIR / f"{VOICE_NAME}.onnx"
CONFIG_PATH = DRIVE_DIR / f"{VOICE_NAME}.onnx.json"
TEST_WAV_PATH = DRIVE_DIR / "teste_castor.wav"

# Disponibiliza opções para células de terminal (%%bash/!).
os.environ["CHECKPOINT_EVERY_N_EPOCHS"] = str(CHECKPOINT_EVERY_N_EPOCHS)

print("Voz:", VOICE_NAME)
print("Épocas adicionais:", ADDITIONAL_EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Pasta persistente:", DRIVE_DIR)


: 

## 2. Confirmar a GPU

O treinamento só deve começar se `nvidia-smi` mostrar uma GPU como **Tesla T4**. A linha `CUDA Version` é do driver; o PyTorch pode usar outra versão CUDA compatível.


In [ ]:
!nvidia-smi


## 3. Montar o Google Drive e localizar o dataset

Na primeira execução, o Google pedirá autorização. Se o ZIP não for encontrado, confira o nome da pasta e do arquivo no bloco de configuração.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

print("Pasta do projeto:", DRIVE_DIR)
print("Dataset esperado:", DATASET_ZIP)
print("Dataset encontrado:", DATASET_ZIP.is_file())

if not DATASET_ZIP.is_file():
    raise FileNotFoundError(
        f"Envie o dataset para {DATASET_ZIP} e execute esta célula novamente."
    )


## 4. Extrair, normalizar e validar o dataset

O dataset é descompactado em `/content` para não treinar lendo milhares de arquivos diretamente do Drive. Esta célula também corrige nomes como `audio\audio_0.wav`, produzidos por alguns ZIPs criados no Windows.

Ela pode ser executada novamente: somente a cópia temporária em `/content/castor_dataset` será recriada.


In [ ]:
import shutil
from pathlib import Path

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

shutil.unpack_archive(str(DATASET_ZIP), str(DATASET_DIR))

metadata_candidates = list(DATASET_DIR.rglob("metadata.csv"))
if not metadata_candidates:
    raise FileNotFoundError("metadata.csv não foi encontrado dentro do ZIP.")

metadata_source = metadata_candidates[0]
if metadata_source.resolve() != METADATA_PATH.resolve():
    shutil.copy2(metadata_source, METADATA_PATH)

# Captura os WAVs antes de criar/normalizar a pasta final.
wav_sources = [
    path
    for path in DATASET_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() == ".wav"
]
if not wav_sources:
    raise FileNotFoundError("Nenhum arquivo WAV foi encontrado dentro do ZIP.")

AUDIO_DIR.mkdir(parents=True, exist_ok=True)

for source in wav_sources:
    # Em Linux, uma barra invertida vinda do Windows pode fazer parte do nome.
    clean_name = Path(source.name.replace("\\", "/")).name
    destination = AUDIO_DIR / clean_name

    if source.resolve() == destination.resolve():
        continue

    if destination.exists():
        if destination.stat().st_size != source.stat().st_size:
            raise FileExistsError(f"Dois WAVs diferentes têm o nome {clean_name}")
        continue

    shutil.move(str(source), str(destination))

with METADATA_PATH.open("r", encoding="utf-8-sig") as metadata_file:
    metadata_lines = [
        line.strip() for line in metadata_file if line.strip()
    ]

invalid_lines = [line for line in metadata_lines if "|" not in line]
if invalid_lines:
    raise ValueError(
        "Existem linhas sem o separador '|'. Exemplos: "
        + repr(invalid_lines[:5])
    )

missing_audio = []
for line in metadata_lines:
    audio_name = Path(line.split("|", 1)[0].replace("\\", "/")).name
    if not (AUDIO_DIR / audio_name).is_file():
        missing_audio.append(audio_name)

wav_count = len(list(AUDIO_DIR.glob("*.wav")))
print("Linhas no metadata:", len(metadata_lines))
print("WAVs normalizados:", wav_count)
print("Áudios não encontrados:", len(missing_audio))
print("Primeiras linhas:")
for line in metadata_lines[:5]:
    print(" ", line)

if missing_audio:
    raise FileNotFoundError(
        "O metadata referencia WAVs ausentes. Exemplos: "
        + repr(missing_audio[:5])
    )

if wav_count != len(metadata_lines):
    print(
        "AVISO: a quantidade de WAVs difere da quantidade de linhas. "
        "O treinamento pode prosseguir se todos os nomes do metadata existem."
    )


## 5. Instalar e compilar o Piper

Esta célula instala as dependências de treinamento e corrige os problemas encontrados no Colab:

- mantém `setuptools < 82`, compatível com o PyTorch da sessão;
- instala `scikit-build`, CMake e Ninja antes da compilação;
- instala os extras de treinamento;
- compila `monotonic_align` e as extensões nativas;
- instala `onnxscript`, exigido pelo PyTorch recente.

Em uma sessão nova, a instalação pode levar vários minutos. Não apague `/content/piper1-gpl` enquanto o diretório atual estiver dentro dele.


In [ ]:
%%bash
set -euo pipefail

cd /content

if [ ! -f /content/piper1-gpl/pyproject.toml ]; then
  rm -rf /content/piper1-gpl
  git clone https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
else
  echo "Repositório Piper já existe; reutilizando a cópia atual."
fi

python -m pip install "setuptools<82" wheel
python -m pip install "scikit-build<1" "cmake>=3.26,<4" "ninja>=1,<2"
python -m pip install -e "/content/piper1-gpl[train]"
python -m pip install onnxscript

cd /content/piper1-gpl
./build_monotonic_align.sh
python setup.py build_ext --inplace


## 6. Verificar Piper, PyTorch e CUDA

O resultado esperado é `CUDA disponível: True`. O caminho do Piper deve apontar para `/content/piper1-gpl/src/piper`.


In [ ]:
import sys

PIPER_SRC = "/content/piper1-gpl/src"
if PIPER_SRC not in sys.path:
    sys.path.insert(0, PIPER_SRC)

import torch
import piper
from piper.train.vits.monotonic_align import maximum_path

print("Piper importado:", piper.__file__)
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA não está disponível. Ative uma GPU no ambiente de execução."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB",
)


## 7. Baixar e verificar o checkpoint brasileiro inicial

O checkpoint Faber Medium é usado somente quando não há um `last.ckpt` da voz Castor ou quando `RESUME_TRAINING = False`. O hash impede que um download incompleto seja usado no treinamento.


In [ ]:
import hashlib
from urllib.request import Request, urlopen

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CHECKPOINT = CHECKPOINT_DIR / "faber-original.ckpt"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as input_file:
        while chunk := input_file.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


needs_download = (
    not BASE_CHECKPOINT.is_file()
    or sha256_file(BASE_CHECKPOINT) != BASE_CHECKPOINT_SHA256
)

if needs_download:
    temporary = BASE_CHECKPOINT.with_suffix(".ckpt.part")
    if temporary.exists():
        temporary.unlink()

    print("Baixando checkpoint Faber...")
    request = Request(BASE_CHECKPOINT_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=180) as response, temporary.open("wb") as out:
        while chunk := response.read(1024 * 1024):
            out.write(chunk)
    temporary.replace(BASE_CHECKPOINT)
else:
    print("Checkpoint original já está no cache local.")

actual_hash = sha256_file(BASE_CHECKPOINT)
print("Arquivo:", BASE_CHECKPOINT)
print("Tamanho:", round(BASE_CHECKPOINT.stat().st_size / 1024**2, 1), "MB")
print("SHA256:", actual_hash)

if actual_hash != BASE_CHECKPOINT_SHA256:
    raise OSError("Hash incorreto: o checkpoint precisa ser baixado novamente.")


## 8. Adaptar o checkpoint antigo para o Lightning atual

O checkpoint oficial possui hiperparâmetros de versões antigas. Esta célula mantém somente os argumentos aceitos pelo `VitsModel` atual. Isso não altera os pesos da voz.


In [ ]:
import inspect
from pathlib import PosixPath

from piper.train.vits.lightning import VitsModel

# Necessário para checkpoints antigos carregados com weights_only=True.
torch.serialization.add_safe_globals([PosixPath])

COMPATIBLE_CHECKPOINT = CHECKPOINT_DIR / "faber-compatible.ckpt"

checkpoint = torch.load(
    BASE_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

valid_parameters = set(inspect.signature(VitsModel.__init__).parameters)
valid_parameters.discard("self")

old_hparams = checkpoint.get("hyper_parameters", {})
checkpoint["hyper_parameters"] = {
    key: value
    for key, value in old_hparams.items()
    if key in valid_parameters and key not in {"dataset", "mos_metric"}
}

torch.save(checkpoint, COMPATIBLE_CHECKPOINT)

print("Checkpoint compatível:", COMPATIBLE_CHECKPOINT)
print("Época base:", checkpoint.get("epoch"))


## 9. Compatibilidade automática ao abrir checkpoints

O Python carrega `sitecustomize.py` automaticamente nos processos de treino e exportação porque adicionaremos `/content` ao `PYTHONPATH`.


In [ ]:
%%writefile /content/sitecustomize.py
from pathlib import PosixPath
import torch

torch.serialization.add_safe_globals([PosixPath])


## 10. Criar o comando de treinamento

Este arquivo usa explicitamente `VitsModel` e `VitsDataModule` — sem isso, versões atuais do Lightning produzem o erro `model.init_args`. Os checkpoints são selecionados por `val_mel` e gravados no Drive no intervalo configurado.


In [ ]:
%%writefile /content/train_castor_colab.py
import os

from lightning.pytorch.callbacks import ModelCheckpoint
from piper.train.__main__ import VitsLightningCLI
from piper.train.vits.dataset import VitsDataModule
from piper.train.vits.lightning import VitsModel

checkpoint_every = int(
    os.environ.get("CHECKPOINT_EVERY_N_EPOCHS", "5")
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_mel",
    mode="min",
    save_top_k=2,
    save_last=True,
    every_n_epochs=checkpoint_every,
    filename="castor-{epoch:04d}-{val_mel:.4f}",
)

VitsLightningCLI(
    VitsModel,
    VitsDataModule,
    save_config_kwargs={"overwrite": True},
    trainer_defaults={"callbacks": [checkpoint_callback]},
)


## 11. Escolher entre iniciar e continuar

- Com `RESUME_TRAINING = True`, o notebook procura o `last.ckpt` mais recente no Drive.
- Se não houver um checkpoint Castor, ele usa o Faber preparado.
- `ADDITIONAL_EPOCHS` sempre significa quantas épocas novas serão executadas nesta rodada.

Exemplo: se o checkpoint terminou na época 6189 e você escolher 50 épocas adicionais, `max_epochs` será calculado automaticamente como 6240.


In [ ]:
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

saved_last_checkpoints = list(
    TRAINING_DIR.glob("lightning_logs/version_*/checkpoints/last.ckpt")
)

if RESUME_TRAINING and saved_last_checkpoints:
    SELECTED_CHECKPOINT = max(
        saved_last_checkpoints,
        key=lambda path: path.stat().st_mtime,
    )
    checkpoint_origin = "último checkpoint Castor"
else:
    SELECTED_CHECKPOINT = COMPATIBLE_CHECKPOINT
    checkpoint_origin = "checkpoint base Faber"

selected_data = torch.load(
    SELECTED_CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)
CURRENT_EPOCH = int(selected_data.get("epoch", -1))
MAX_EPOCHS = CURRENT_EPOCH + 1 + ADDITIONAL_EPOCHS

del selected_data

os.environ["CASTOR_CHECKPOINT"] = str(SELECTED_CHECKPOINT)
os.environ["CASTOR_MAX_EPOCHS"] = str(MAX_EPOCHS)
os.environ["CASTOR_BATCH_SIZE"] = str(BATCH_SIZE)
os.environ["CASTOR_TRAINING_DIR"] = str(TRAINING_DIR)
os.environ["CASTOR_CONFIG_PATH"] = str(CONFIG_PATH)
os.environ["CHECKPOINT_EVERY_N_EPOCHS"] = str(CHECKPOINT_EVERY_N_EPOCHS)

print("Origem:", checkpoint_origin)
print("Checkpoint:", SELECTED_CHECKPOINT)
print("Época atual:", CURRENT_EPOCH)
print("Épocas novas:", ADDITIONAL_EPOCHS)
print("max_epochs calculado:", MAX_EPOCHS)


## 12. Iniciar o treinamento

A primeira execução processa os WAVs e cria `/content/castor_cache`; durante essa fase a GPU pode ficar ociosa. Depois surgirão as barras de `Epoch`.

Se ocorrer `CUDA out of memory`, interrompa, altere `BATCH_SIZE` para 4 no bloco de configuração e execute novamente as células 11 e 12.


In [ ]:
!PYTHONPATH=/content:/content/piper1-gpl/src \
python /content/train_castor_colab.py fit \
  --trainer.accelerator gpu \
  --trainer.devices 1 \
  --trainer.precision 16-mixed \
  --trainer.max_epochs "$CASTOR_MAX_EPOCHS" \
  --trainer.default_root_dir "$CASTOR_TRAINING_DIR" \
  --trainer.log_every_n_steps 10 \
  --trainer.num_sanity_val_steps 0 \
  --model.sample_rate 22050 \
  --model.mos_metric none \
  --data.voice_name castor \
  --data.csv_path /content/castor_dataset/metadata.csv \
  --data.audio_dir /content/castor_dataset/audio \
  --data.espeak_voice pt-br \
  --data.cache_dir /content/castor_cache \
  --data.config_path "$CASTOR_CONFIG_PATH" \
  --data.batch_size "$CASTOR_BATCH_SIZE" \
  --data.num_workers 2 \
  --data.validation_split 0.02 \
  --ckpt_path "$CASTOR_CHECKPOINT"


### Acompanhar a GPU durante o treinamento

Abra o painel **Terminal** do Colab e execute:

```bash
watch -n 2 nvidia-smi
```

Para encerrar o monitor, pressione `Ctrl+C`. É normal o uso cair durante validação, carregamento dos dados e gravação de checkpoints.

O treinamento terminou corretamente quando aparecer algo parecido com:

```text
Trainer.fit stopped: max_epochs=... reached
```


## 13. Conferir os checkpoints salvos

Execute esta célula depois que o treinamento terminar. O `last.ckpt` é o ponto recomendado para continuar em outra sessão.


In [ ]:
all_checkpoints = sorted(
    TRAINING_DIR.glob("lightning_logs/version_*/checkpoints/*.ckpt"),
    key=lambda path: path.stat().st_mtime,
)

if not all_checkpoints:
    raise FileNotFoundError("Nenhum checkpoint foi salvo no Google Drive.")

print("Checkpoints encontrados:")
for path in all_checkpoints:
    print(
        " -",
        path.relative_to(DRIVE_DIR),
        f"({path.stat().st_size / 1024**2:.1f} MB)",
    )


## Como continuar por mais épocas depois

1. Altere `ADDITIONAL_EPOCHS` no bloco 1, por exemplo para 50.
2. Deixe `RESUME_TRAINING = True`.
3. Em uma sessão que ainda está aberta, execute novamente a partir do bloco 11.
4. Em uma sessão nova, execute o notebook desde o início; o Drive preserva o `last.ckpt`.

Treine em blocos de 30–50 épocas e compare sempre a mesma frase. Isso facilita perceber melhora ou sobreajuste.


## 14. Preparar o exportador ONNX

Versões recentes do PyTorch usam por padrão o novo exportador baseado em Dynamo. O VITS do Piper possui controle de fluxo que pode causar `GuardOnDataDependentSymNode`. Esta célula força o exportador clássico com `dynamo=False`.


In [ ]:
exporter_path = PIPER_DIR / "src" / "piper" / "train" / "export_onnx.py"
exporter_code = exporter_path.read_text(encoding="utf-8")

if "dynamo=False" not in exporter_code:
    marker = "        dynamic_axes={"
    if marker not in exporter_code:
        raise RuntimeError(
            "A estrutura de export_onnx.py mudou e o ponto de correção não foi encontrado."
        )

    exporter_code = exporter_code.replace(
        marker,
        "        dynamo=False,\n" + marker,
        1,
    )
    exporter_path.write_text(exporter_code, encoding="utf-8")
    print("Exportador configurado com dynamo=False.")
else:
    print("Exportador já estava configurado com dynamo=False.")


## 15. Selecionar o último checkpoint e exportar

A exportação usa o `last.ckpt` mais recente do Drive, independentemente de qual versão do `lightning_logs` o contém.


In [ ]:
latest_last_candidates = list(
    TRAINING_DIR.glob("lightning_logs/version_*/checkpoints/last.ckpt")
)
if not latest_last_candidates:
    raise FileNotFoundError("Nenhum last.ckpt foi encontrado para exportação.")

EXPORT_CHECKPOINT = max(
    latest_last_candidates,
    key=lambda path: path.stat().st_mtime,
)

os.environ["CASTOR_EXPORT_CHECKPOINT"] = str(EXPORT_CHECKPOINT)
os.environ["CASTOR_MODEL_PATH"] = str(MODEL_PATH)

print("Checkpoint que será exportado:", EXPORT_CHECKPOINT)
print("Modelo de destino:", MODEL_PATH)


In [ ]:
!cd /content/piper1-gpl && \
PYTHONPATH=/content:/content/piper1-gpl/src \
python -m piper.train.export_onnx \
  --checkpoint "$CASTOR_EXPORT_CHECKPOINT" \
  --output-file "$CASTOR_MODEL_PATH"


## 16. Validar os arquivos finais

O Piper precisa dos dois arquivos com o mesmo nome-base:

- `pt_BR-castor-medium.onnx`
- `pt_BR-castor-medium.onnx.json`


In [ ]:
import json
import onnx

print("Modelo ONNX:", MODEL_PATH.is_file())
print("Configuração:", CONFIG_PATH.is_file())

if not MODEL_PATH.is_file():
    raise FileNotFoundError(MODEL_PATH)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)

with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    voice_config = json.load(config_file)

onnx.checker.check_model(str(MODEL_PATH))

print("ONNX válido!")
print("Tamanho:", round(MODEL_PATH.stat().st_size / 1024**2, 1), "MB")
print("Sample rate:", voice_config.get("audio", {}).get("sample_rate"))
print("Voz espeak:", voice_config.get("espeak", {}).get("voice"))


## 17. Gerar e ouvir um WAV de teste

Os três parâmetros de síntese ficam no bloco de configuração. Para uma saída mais suave, experimente `noise_scale=0.50` e `noise_w_scale=0.60`. `length_scale` menor fala mais rápido; maior fala mais devagar.


In [ ]:
import wave
from IPython.display import Audio, display
from piper import PiperVoice, SynthesisConfig

voice = PiperVoice.load(
    str(MODEL_PATH),
    config_path=str(CONFIG_PATH),
)

synthesis_config = SynthesisConfig(
    noise_scale=TEST_NOISE_SCALE,
    length_scale=TEST_LENGTH_SCALE,
    noise_w_scale=TEST_NOISE_W_SCALE,
)

with wave.open(str(TEST_WAV_PATH), "wb") as wav_file:
    voice.synthesize_wav(
        TEST_TEXT,
        wav_file,
        syn_config=synthesis_config,
    )

print("Áudio gerado:", TEST_WAV_PATH)
display(Audio(str(TEST_WAV_PATH)))


## 18. Baixar os arquivos para o computador (opcional)

Eles já estão seguros no Google Drive. Execute uma linha por vez se também quiser baixá-los pelo navegador.


In [ ]:
from google.colab import files

# Descomente UMA linha por vez para baixar:
# files.download(str(MODEL_PATH))
# files.download(str(CONFIG_PATH))
# files.download(str(TEST_WAV_PATH))


## Solução rápida de problemas

### `Dataset encontrado: False`
O ZIP não está em `Meu Drive/piper-castor/castor_dataset.zip`. Ajuste `DRIVE_FOLDER` ou `DATASET_ZIP_NAME` no bloco 1.

### `WAVs: 0` ou nomes `audio\\audio_0.wav`
Execute novamente o bloco 4; ele normaliza as barras invertidas do Windows.

### `No module named skbuild`
Execute novamente o bloco 5. O pacote correto no `pip` chama-se `scikit-build`.

### `No module named piper`
Execute os blocos 5 e 6. O notebook adiciona `/content/piper1-gpl/src` ao caminho.

### `Target key expected to start with model.init_args`
O script correto está no bloco 10 e passa explicitamente `VitsModel` e `VitsDataModule`.

### `CUDA out of memory`
Troque `BATCH_SIZE = 8` por `BATCH_SIZE = 4` e retome do último checkpoint.

### `GuardOnDataDependentSymNode` na exportação
Execute novamente o bloco 14 para aplicar `dynamo=False`.

### A sessão do Colab foi desconectada
Execute o notebook novamente desde o início. O dataset ZIP e os checkpoints permanecem no Drive. Com `RESUME_TRAINING = True`, o último `last.ckpt` será selecionado automaticamente.

### A voz ficou picotada, metálica ou pouco parecida
Primeiro teste valores menores de `TEST_NOISE_SCALE` e `TEST_NOISE_W_SCALE`. Se ainda estiver imatura, continue em blocos de 30–50 épocas e compare checkpoints. A qualidade e o alinhamento entre cada WAV e sua frase são tão importantes quanto o número de épocas.
